# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/franciskendrick/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [4]:
import duckdb
from google.colab import userdata

# Initialize DuckDB & Authenticate via Secret Manager
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

WAREHOUSE_URI = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH_PATH = f"{WAREHOUSE_URI}/fact_content_daily_performance/month=2026-03/*.parquet"

query_signal_audit = f"""
SELECT
    content_hash_id,

    -- Corrected Column Binding & Weighted Position Aggregation
    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks,

    -- Impression-Weighted Average Position (Statistically Sound)
    ROUND(
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0), 2
    ) AS weighted_avg_position,

    -- Unweighted Daily Average Position (For Comparison Only)
    ROUND(AVG(gsc_avg_position), 2) AS unweighted_avg_position,

    -- Overall Click-Through Rate
    ROUND(
        100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 2
    ) AS ctr_pct

FROM read_parquet('{FACT_MARCH_PATH}')
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
  AND gsc_data_available IS TRUE
GROUP BY content_hash_id
HAVING SUM(gsc_impressions) > 100
ORDER BY total_impressions DESC
LIMIT 10;
"""

df_signal = con.execute(query_signal_audit).df()
print("--- SIGNAL CHECK 1: CORRECTED POSITION AGGREGATION ---")
print(df_signal.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- SIGNAL CHECK 1: CORRECTED POSITION AGGREGATION ---
         content_hash_id  total_impressions  total_clicks  weighted_avg_position  unweighted_avg_position  ctr_pct
content_eadb33b5df496f4a           617124.0        5668.0                   2.33                     2.38     0.92
content_ec2e0346994fb5a5           245276.0        1480.0                   2.76                     2.85     0.60
content_e8a52cf3d5988c07           244931.0         669.0                  15.17                    15.01     0.27
content_0e03de7680314cd5           221310.0         720.0                   2.51                     2.68     0.33
content_44f34c0a90047651           212404.0          24.0                   0.67                     7.35     0.01
content_7172a7fad43f0998           205867.0         862.0                   3.30                     3.37     0.42
content_e7b5dd4dff461ad2           205045.0        2446.0                   4.45                     4.54     1.19
content_8d7d99f109e19aa2 

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.